# XAI stability -- TESTS

In [ ]:
import sys

!{sys.executable} -m pip install nbimporter
!{sys.executable} -m pip install tensorflow
!{sys.executable} -m pip install torch

#!git clone https://github.com/AI4LIFE-GROUP/OpenXAI.git
!{sys.executable} -m pip install -e OpenXAI

In [2]:
import time
import numpy as np
import pandas as pd
import nbimporter

# Utils
import torch
import os
import pickle
from sklearn.base import clone

import xgboost as xgb
from sklearn.neural_network import MLPClassifier


import Taylor_Explainer as texp
import XAI_stability_metrics as stab


import sklearn.ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [3]:
import openxai

# Data loaders
from openxai.dataloader import return_loaders

# Perturbation methods required for the computation of the relative stability metrics
from openxai.explainers.catalog.perturbation_methods import NormalPerturbation
from openxai.explainers.catalog.perturbation_methods import NewDiscrete_NormalPerturbation

# Perturbation definition

In [4]:
# Perturbation class parameters
perturbation_mean= 0.0
perturbation_std= 0.05
perturbation_flip_percentage= 0.01
    
perturbation= NormalPerturbation('tabular',
                                 mean=perturbation_mean,
                                 std_dev=perturbation_std,
                                 flip_percentage=perturbation_flip_percentage)

def generate_mask(explanation, top_k):
    mask_indices= torch.topk(explanation, top_k).indices
    mask= torch.zeros(explanation.shape) > 10
    for i in mask_indices:
        mask[i]= True
    return mask

# Loading data

# 1. Synthetic - 20 features - Numeric

In [5]:
ox_path= 'data/synth_OX_20/processed/'

train_ox= pd.read_csv(ox_path + 'X_train.csv')
test_ox = pd.read_csv(ox_path + 'X_test.csv')
labels_train_ox= pd.read_csv(ox_path + 'y_train.csv')
labels_test_ox = pd.read_csv(ox_path + 'y_test.csv')

In [6]:
# 2024-05-29 23:24:10,338 Best: 0.908399 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_ox= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=256,
                            hidden_layer_sizes=(256,),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_ox.fit(train_ox, labels_train_ox.values.ravel())

acc_nn1_ox= sklearn.metrics.accuracy_score(labels_test_ox.values.ravel(), nn1_model_ox.predict(test_ox))
acc_nn1_ox

0.83

In [7]:
# 2024-05-31 17:29:14,888 Best: 0.912651 using {'batch_size': 64, 'lr': 0.01, 'max_epochs': 64, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_ox= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_ox.fit(train_ox, labels_train_ox.values.ravel())

acc_nn2_ox= sklearn.metrics.accuracy_score(labels_test_ox.values.ravel(), nn2_model_ox.predict(test_ox))
acc_nn2_ox

0.83

In [8]:
# 2024-06-03 16:27:56,409 Best: 0.905554 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_ox= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_ox.fit(train_ox, labels_train_ox.values.ravel())

acc_nn3_ox= sklearn.metrics.accuracy_score(labels_test_ox.values.ravel(), nn3_model_ox.predict(test_ox))
acc_nn3_ox

0.84

In [7]:
# definitions ---- 154 samples from test dataset

# get n and m parameters from train and labels_train
n_ox, m_ox= texp.get_n_m_sizes(test_ox.loc[0:153], labels_test_ox[0:153])

# conversion of train_ox and labels_train_ox data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_ox.values)
tn_lb_tr= torch.from_numpy(labels_train_ox.values.ravel().astype(int))


# define a descriptor to synthetic data
descriptor_ox= dict()

h_min_dist_ox= texp.get_minimum_distance(train_ox)

# T-Exp explanation settings
descriptor_ox['h_min']= h_min_dist_ox
descriptor_ox['h_max']= 1
descriptor_ox['jacobian_eps']= 1e-3
descriptor_ox['max_itr']= 30
descriptor_ox['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_ox['num_samples']= 30
descriptor_ox['num_perts']= 10   # RIS/ROS
descriptor_ox['pert_max_distance']= (h_min_dist_ox/2)
descriptor_ox['num_runs']= 10    # RES
descriptor_ox['feature_metadata']= ['c'] * n_ox
descriptor_ox['p_norm']= 2
descriptor_ox['eps_norm']= 1e-6
descriptor_ox['top_k']= 0
descriptor_ox['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_ox['top_k'])

In [46]:
# ---- 154 samples from test dataset

# evaluate relative input/output stability -- using the 2 first instances from train_ox
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_ox= stab.relative_stability(nn1_model_ox, test_ox[0:153], labels_test_ox[0:153], perturbation, 
                                        descriptor_ox, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

RIS/ROS --
Instance 1 out 153


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


...

  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]


--- 7616.66 seconds ---


Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


In [47]:
ris_ros_nn1_ox

`should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.


{'t_exp_ris_max': 5603892.524780785,
 'std(t_exp_ris_max)': 493187.94537293626,
 'shap_ris_max': 229068.439761331,
 'std(shap_ris_max)': 29150.10027726519,
 'lime_ris_max': 817.7087942454799,
 'std(lime_ris_max)': 91.59088404931305,
 't_exp_ris_mean': 18563.799530082037,
 'std(t_exp_ris_mean)': 178969.49702517316,
 'shap_ris_mean': 3190.4811885037684,
 'std(shap_ris_mean)': 10800.258471750327,
 'lime_ris_mean': 8.215667449593752,
 'std(lime_ris_mean)': 35.67043482243291,
 't_exp_ros_max': 2458674148301.455,
 'std(t_exp_ros_max)': 199440013418.45062,
 'shap_ros_max': 25000002459.684566,
 'std(shap_ros_max)': 2344129117.3748603,
 'lime_ros_max': 4389568.240176316,
 'std(lime_ros_max)': 519615.6495410408,
 't_exp_ros_mean': 4520681840.164775,
 'std(t_exp_ros_mean)': 50893715943.55178,
 'shap_ros_mean': 77259304.44405259,
 'std(shap_ros_mean)': 557800015.4149939,
 'lime_ros_mean': 35419.93134790406,
 'std(lime_ros_mean)': 173723.09099071677,
 'shap_kernel_ris_max': 607.0795586674479,
 'std

In [48]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ox
start= time.time()
print('RES --')
res_nn1_ox= stab.run_stability(nn1_model_ox, test_ox[0:153], labels_test_ox[0:153], 
                               descriptor_ox, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

RES --
Instance 1 out 153


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


...

  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]


--- 6744.26 seconds ---


Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


In [49]:
res_nn1_ox

`should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.


{'texp_res': 9.455164053283262e-15,
 'shap_res': 0.1634944570925877,
 'shap_kernel_res': 0.04685447237845159,
 'shap_exact_res': '--',
 'lime_res': 5.801606246868625e-17,
 'itGd_res': 1.1149867635186251e-14,
 'iXGd_res': 8.145812e-06,
 'dLif_res': 8.302823e-06,
 'lwrp_res': 8.777078e-06,
 'smoothG_res': 8.466344434769473,
 'vanillaG_res': 1.3662861e-05,
 'GuidBprop_res': 1.3662861e-05,
 'occlusion_res': 5.2452087e-06}

In [50]:
# evaluate relative input/output stability -- using the 2 first instances from train_ox
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_ox= stab.relative_stability(nn2_model_ox, test_ox[0:153], labels_test_ox[0:153], perturbation, 
                                        descriptor_ox, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

RIS/ROS --
Instance 1 out 153


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


...

  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]


--- 13982.16 seconds ---


Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


In [51]:
ris_ros_nn2_ox

`should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.


{'t_exp_ris_max': 7640.491969322056,
 'std(t_exp_ris_max)': 647.2195860582013,
 'shap_ris_max': 270095.48418185214,
 'std(shap_ris_max)': 24991.794537847083,
 'lime_ris_max': 484.68645783727106,
 'std(lime_ris_max)': 42.79695885530299,
 't_exp_ris_mean': 65.59534800672762,
 'std(t_exp_ris_mean)': 404.64275803562134,
 'shap_ris_mean': 2359.9648125921053,
 'std(shap_ris_mean)': 7853.573476711983,
 'lime_ris_mean': 4.675652331847907,
 'std(lime_ris_mean)': 17.658517326843477,
 't_exp_ros_max': 41628.19390976018,
 'std(t_exp_ros_max)': 5197.718072706151,
 'shap_ros_max': 553754.4173019871,
 'std(shap_ros_max)': 65699.11998168417,
 'lime_ros_max': 86252.34339146587,
 'std(lime_ros_max)': 6961.734945763653,
 't_exp_ros_mean': 190.7131194183298,
 'std(t_exp_ros_mean)': 691.260499091987,
 'shap_ros_mean': 4024.5761584057063,
 'std(shap_ros_mean)': 15739.178387958023,
 'lime_ros_mean': 68.25591118854437,
 'std(lime_ros_mean)': 699.818028083017,
 'shap_kernel_ris_max': 7549.4556812145565,
 'std(

In [10]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ox
start= time.time()
print('RES --')
res_nn2_ox= stab.run_stability(nn2_model_ox, test_ox[0:153], labels_test_ox[0:153], 
                               descriptor_ox, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

RES --
Instance 0 out 153


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


...

  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]


--- 12459.28 seconds ---


Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


In [11]:
res_nn2_ox

`should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.


{'texp_res': 4.0943002132167226e-15,
 'shap_res': 0.15390322085287525,
 'shap_kernel_res': 0.04867681553833151,
 'shap_exact_res': '--',
 'lime_res': 5.3617058706823765e-17,
 'itGd_res': 7.838739178637249e-15,
 'iXGd_res': 3.1411776e-06,
 'dLif_res': 3.1047741e-06,
 'lwrp_res': 2.6744574e-06,
 'smoothG_res': 4.171694414679018,
 'vanillaG_res': 7.0414253e-06,
 'GuidBprop_res': 7.0414253e-06,
 'occlusion_res': 2.311554e-06}

In [12]:
# evaluate relative input/output stability -- using the 2 first instances from train_ox
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_ox= stab.relative_stability(nn3_model_ox, test_ox[0:153], labels_test_ox[0:153], perturbation, 
                                        descriptor_ox, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

RIS/ROS --
Instance 0 out 153


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


...

  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished



--- 17001.97 seconds ---


In [13]:
ris_ros_nn3_ox

`should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.


{'t_exp_ris_max': 4077.5863498180615,
 'std(t_exp_ris_max)': 344.4483969619061,
 'shap_ris_max': 299147.8385361835,
 'std(shap_ris_max)': 31433.977843292447,
 'lime_ris_max': 311.68267079056506,
 'std(lime_ris_max)': 25.132565950835794,
 't_exp_ris_mean': 33.35642342725415,
 'std(t_exp_ris_mean)': 111.89416953730306,
 'shap_ris_mean': 3199.5807244478146,
 'std(shap_ris_mean)': 14374.932617042905,
 'lime_ris_mean': 2.050617331314682,
 'std(lime_ris_mean)': 13.523956207570828,
 't_exp_ros_max': 124006.89366487059,
 'std(t_exp_ros_max)': 10149.733107955286,
 'shap_ros_max': 18818810.87146958,
 'std(shap_ros_max)': 1529281.4990617502,
 'lime_ros_max': 2154.2681596059538,
 'std(lime_ros_max)': 210.56655111518265,
 't_exp_ros_mean': 263.5036106125044,
 'std(t_exp_ros_mean)': 2004.9551345042628,
 'shap_ros_mean': 32316.33154762191,
 'std(shap_ros_mean)': 336051.5893422897,
 'lime_ros_mean': 8.491760012174298,
 'std(lime_ros_mean)': 36.82913716206885,
 'shap_kernel_ris_max': 229.64425771475123

In [14]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ox
start= time.time()
print('RES --')
res_nn3_ox= stab.run_stability(nn3_model_ox, test_ox[0:153], labels_test_ox[0:153], 
                               descriptor_ox, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

RES --
Instance 0 out 153


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


...

  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished



--- 14545.86 seconds ---


In [15]:
res_nn3_ox

`should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.


{'texp_res': 5.630757419431282e-15,
 'shap_res': 0.14022637947262276,
 'shap_kernel_res': 0.04919905163070115,
 'shap_exact_res': '--',
 'lime_res': 5.793820594181519e-17,
 'itGd_res': 5.208590938883218e-15,
 'iXGd_res': 4.5775437e-06,
 'dLif_res': 4.874844e-06,
 'lwrp_res': 4.572496e-06,
 'smoothG_res': 5.46918516902664,
 'vanillaG_res': 7.6779015e-06,
 'GuidBprop_res': 7.6779015e-06,
 'occlusion_res': 2.5788913e-06}

# TODO LIST
# - Setar parâmetros dos modelos de 2., 3., 8., 10.
# - Verificar h_min de todos os conjuntos antes de testar e fixar um h_mim, caso necessário

# 2. Adult Income

In [6]:
ad_path= 'data/adult/processed/'

train_ad= pd.read_csv(ad_path + 'X_train.csv')
test_ad = pd.read_csv(ad_path + 'X_test.csv')
labels_train_ad= pd.read_csv(ad_path + 'y_train.csv')
labels_test_ad = pd.read_csv(ad_path + 'y_test.csv')

In [ ]:
# 2024-05-29 23:24:10,338 Best: 0.908399 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_ad= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=256,
                            hidden_layer_sizes=(256,),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_ad.fit(train_ad, labels_train_ad.values.ravel())

acc_nn1_ad= sklearn.metrics.accuracy_score(labels_test_ad.values.ravel(), nn1_model_ad.predict(test_ad))
acc_nn1_ad

In [ ]:
# 2024-05-31 17:29:14,888 Best: 0.912651 using {'batch_size': 64, 'lr': 0.01, 'max_epochs': 64, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_ad= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_ad.fit(train_ad, labels_train_ad.values.ravel())

acc_nn2_ad= sklearn.metrics.accuracy_score(labels_test_ad.values.ravel(), nn2_model_ad.predict(test_ad))
acc_nn2_ad

In [ ]:
# 2024-06-03 16:27:56,409 Best: 0.905554 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_ad= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_ad.fit(train_ad, labels_train_ad.values.ravel())

acc_nn3_ad= sklearn.metrics.accuracy_score(labels_test_ad.values.ravel(), nn3_model_ad.predict(test_ad))
acc_nn3_ad

In [ ]:
# definitions ---- 624 samples from test dataset

# get n and m parameters from train and labels_train
n_ad, m_ad= texp.get_n_m_sizes(test_ad.loc[0:623], labels_test_ad[0:623])

# conversion of train_ad and labels_train_ad data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_ad.values)
tn_lb_tr= torch.from_numpy(labels_train_ad.values.ravel().astype(int))


# define a descriptor to synthetic data
descriptor_ad= dict()

h_min_dist_ad= texp.get_minimum_distance(train_ad)

# T-Exp explanation settings
descriptor_ad['h_min']= h_min_dist_ad
descriptor_ad['h_max']= 1
descriptor_ad['jacobian_eps']= 1e-3
descriptor_ad['max_itr']= 30
descriptor_ad['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_ad['num_samples']= 30
descriptor_ad['num_perts']= 10   # RIS/ROS
descriptor_ad['pert_max_distance']= (h_min_dist_ad/2)
descriptor_ad['num_runs']= 10    # RES
descriptor_ad['feature_metadata']= ['c'] * n_ad
descriptor_ad['p_norm']= 2
descriptor_ad['eps_norm']= 1e-6
descriptor_ad['top_k']= 0
descriptor_ad['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_ad['top_k'])

In [ ]:
# ---- 624 samples from test dataset

# evaluate relative input/output stability -- using the 2 first instances from train_ad
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_ad= stab.relative_stability(nn1_model_ad, test_ad[0:623], labels_test_ad[0:623], perturbation, 
                                        descriptor_ad, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn1_ad

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ad
start= time.time()
print('RES --')
res_nn1_ad= stab.run_stability(nn1_model_ad, test_ad[0:623], labels_test_ad[0:623], 
                               descriptor_ad, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn1_ad

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_ad
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_ad= stab.relative_stability(nn2_model_ad, test_ad[0:623], labels_test_ad[0:623], perturbation, 
                                        descriptor_ad, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn2_ad

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ad
start= time.time()
print('RES --')
res_nn2_ad= stab.run_stability(nn2_model_ad, test_ad[0:623], labels_test_ad[0:623], 
                               descriptor_ad, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn2_ad

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_ad
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_ad= stab.relative_stability(nn3_model_ad, test_ad[0:623], labels_test_ad[0:623], perturbation, 
                                        descriptor_ad, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn3_ad

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ad
start= time.time()
print('RES --')
res_nn3_ad= stab.run_stability(nn3_model_ad, test_ad[0:623], labels_test_ad[0:623], 
                               descriptor_ad, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn3_ad

# 3. Chess (kr-vs-kp)

In [7]:
ch_path= 'data/chess/processed/'

train_ch= pd.read_csv(ch_path + 'X_train.csv')
test_ch = pd.read_csv(ch_path + 'X_test.csv')
labels_train_ch= pd.read_csv(ch_path + 'y_train.csv')
labels_test_ch = pd.read_csv(ch_path + 'y_test.csv')

In [ ]:
# 2024-05-29 23:24:10,338 Best: 0.908399 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_ch= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=256,
                            hidden_layer_sizes=(256,),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_ch.fit(train_ch, labels_train_ch.values.ravel())

acc_nn1_ch= sklearn.metrics.accuracy_score(labels_test_ch.values.ravel(), nn1_model_ch.predict(test_ch))
acc_nn1_ch

In [ ]:
# 2024-05-31 17:29:14,888 Best: 0.912651 using {'batch_size': 64, 'lr': 0.01, 'max_epochs': 64, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_ch= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_ch.fit(train_ch, labels_train_ch.values.ravel())

acc_nn2_ch= sklearn.metrics.accuracy_score(labels_test_ch.values.ravel(), nn2_model_ch.predict(test_ch))
acc_nn2_ch

In [ ]:
# 2024-06-03 16:27:56,409 Best: 0.905554 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_ch= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_ch.fit(train_ch, labels_train_ch.values.ravel())

acc_nn3_ch= sklearn.metrics.accuracy_score(labels_test_ch.values.ravel(), nn3_model_ch.predict(test_ch))
acc_nn3_ch

In [ ]:
# definitions ---- 327 samples from test dataset

# get n and m parameters from train and labels_train
n_ch, m_ch= texp.get_n_m_sizes(test_ch.loc[0:326], labels_test_ch[0:326])

# conversion of train_ch and labels_train_ch data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_ch.values)
tn_lb_tr= torch.from_numpy(labels_train_ch.values.ravel().astype(int))


# define a descriptor to synthetic data
descriptor_ch= dict()

h_min_dist_ch= texp.get_minimum_distance(train_ch)

# T-Exp explanation settings
descriptor_ch['h_min']= h_min_dist_ch
descriptor_ch['h_max']= 1
descriptor_ch['jacobian_eps']= 1e-3
descriptor_ch['max_itr']= 30
descriptor_ch['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_ch['num_samples']= 30
descriptor_ch['num_perts']= 10   # RIS/ROS
descriptor_ch['pert_max_distance']= (h_min_dist_ch/2)
descriptor_ch['num_runs']= 10    # RES
descriptor_ch['feature_metadata']= ['c'] * n_ch
descriptor_ch['p_norm']= 2
descriptor_ch['eps_norm']= 1e-6
descriptor_ch['top_k']= 0
descriptor_ch['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_ch['top_k'])

In [ ]:
# ---- 327 samples from test dataset

# evaluate relative input/output stability -- using the 2 first instances from train_ch
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_ch= stab.relative_stability(nn1_model_ch, test_ch[0:326], labels_test_ch[0:326], perturbation, 
                                        descriptor_ch, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn1_ch

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ch
start= time.time()
print('RES --')
res_nn1_ch= stab.run_stability(nn1_model_ch, test_ch[0:326], labels_test_ch[0:326], 
                               descriptor_ch, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn1_ch

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_ch
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_ch= stab.relative_stability(nn2_model_ch, test_ch[0:326], labels_test_ch[0:326], perturbation, 
                                        descriptor_ch, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn2_ch

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ch
start= time.time()
print('RES --')
res_nn2_ch= stab.run_stability(nn2_model_ch, test_ch[0:326], labels_test_ch[0:326], 
                               descriptor_ch, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn2_ch

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_ch
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_ch= stab.relative_stability(nn3_model_ch, test_ch[0:326], labels_test_ch[0:326], perturbation, 
                                        descriptor_ch, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn3_ch

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ch
start= time.time()
print('RES --')
res_nn3_ch= stab.run_stability(nn3_model_ch, test_ch[0:326], labels_test_ch[0:326], 
                               descriptor_ch, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn3_ch

# 4. COMPAS

In [8]:
cpas_path= 'data/compas/processed/'

train_cpas= pd.read_csv(cpas_path + 'X_train.csv')
test_cpas = pd.read_csv(cpas_path + 'X_test.csv')
labels_train_cpas= pd.read_csv(cpas_path + 'y_train.csv')
labels_test_cpas = pd.read_csv(cpas_path + 'y_test.csv')

In [ ]:
# 2024-05-30 04:34:50,307 Best: 0.729593 using {'batch_size': 32, 'lr': 0.01, 'max_epochs': 128, 
#                               'module__n_features': 13, 'module__n_neurons': 16, 'module__nonlin': Tanh()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_cpas= MLPClassifier(batch_size= 32,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(16,),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_cpas.fit(train_cpas, labels_train_cpas.values.ravel())

acc_nn1_cpas= sklearn.metrics.accuracy_score(labels_test_cpas.values.ravel(), nn1_model_cpas.predict(test_cpas))
acc_nn1_cpas

In [ ]:
# 2024-05-31 23:28:32,931 Best: 0.728194 using {'batch_size': 32, 'lr': 0.02, 'max_epochs': 32, 
#                               'module__n_features': 13, 'module__n_neurons': 32, 'module__nonlin': Tanh()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_cpas= MLPClassifier(batch_size= 32,
                            learning_rate='constant', learning_rate_init= 0.02,
                            max_iter=32,
                            hidden_layer_sizes=(32, 32),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_cpas.fit(train_cpas, labels_train_cpas.values.ravel())

acc_nn2_cpas= sklearn.metrics.accuracy_score(labels_test_cpas.values.ravel(), nn2_model_cpas.predict(test_cpas))
acc_nn2_cpas

In [ ]:
# 2024-06-03 22:58:26,596 Best: 0.728264 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 32, 
#                               'module__n_features': 13, 'module__n_neurons': 16, 'module__nonlin': Tanh()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_cpas= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=32,
                            hidden_layer_sizes=(16, 16, 16),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_cpas.fit(train_cpas, labels_train_cpas.values.ravel())

acc_nn3_cpas= sklearn.metrics.accuracy_score(labels_test_cpas.values.ravel(), nn3_model_cpas.predict(test_cpas))
acc_nn3_cpas

In [ ]:
# definitions ---- 409 samples from test dataset

# get n and m parameters from train and labels_train
n_cpas, m_cpas= texp.get_n_m_sizes(test_cpas.loc[0:408], labels_test_cpas[0:408])

# conversion of train_cpas and labels_train_cpas data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_cpas.values)
tn_lb_tr= torch.from_numpy(labels_train_cpas.values.ravel().astype(int))


# define a descriptor to synthetic data
descriptor_cpas= dict()

h_min_dist_cpas= texp.get_minimum_distance(train_cpas)

# T-Exp explanation settings
descriptor_cpas['h_min']= h_min_dist_cpas
descriptor_cpas['h_max']= 1
descriptor_cpas['jacobian_eps']= 1e-3
descriptor_cpas['max_itr']= 30
descriptor_cpas['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_cpas['num_samples']= 30
descriptor_cpas['num_perts']= 10   # RIS/ROS
descriptor_cpas['pert_max_distance']= (h_min_dist_cpas/2)
descriptor_cpas['num_runs']= 10    # RES
descriptor_cpas['feature_metadata']= ['c'] * n_cpas
descriptor_cpas['p_norm']= 2
descriptor_cpas['eps_norm']= 1e-6
descriptor_cpas['top_k']= 0
descriptor_cpas['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_cpas['top_k'])

In [ ]:
# ---- 409 samples from test dataset

# evaluate relative input/output stability -- using the 2 first instances from train_cpas
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_cpas= stab.relative_stability(nn1_model_cpas, test_cpas[0:408], labels_test_cpas[0:408], perturbation, 
                                        descriptor_cpas, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn1_cpas

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_cpas
start= time.time()
print('RES --')
res_nn1_cpas= stab.run_stability(nn1_model_cpas, test_cpas[0:408], labels_test_cpas[0:408], 
                               descriptor_cpas, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn1_cpas

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_cpas
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_cpas= stab.relative_stability(nn2_model_cpas, test_cpas[0:408], labels_test_cpas[0:408], perturbation, 
                                        descriptor_cpas, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn2_cpas

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_cpas
start= time.time()
print('RES --')
res_nn2_cpas= stab.run_stability(nn2_model_cpas, test_cpas[0:408], labels_test_cpas[0:408], 
                               descriptor_cpas, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn2_cpas

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_cpas
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_cpas= stab.relative_stability(nn3_model_cpas, test_cpas[0:408], labels_test_cpas[0:408], perturbation, 
                                        descriptor_cpas, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn3_cpas

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_cpas
start= time.time()
print('RES --')
res_nn3_cpas= stab.run_stability(nn3_model_cpas, test_cpas[0:408], labels_test_cpas[0:408], 
                               descriptor_cpas, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn3_cpas

# 5. Diabetes

In [47]:
diab_path= 'data/diabetes/processed/'

train_diab= pd.read_csv(diab_path + 'X_train.csv')
test_diab = pd.read_csv(diab_path + 'X_test.csv')
labels_train_diab= pd.read_csv(diab_path + 'y_train.csv')
labels_test_diab = pd.read_csv(diab_path + 'y_test.csv')

In [49]:
# 2024-05-29 17:25:01,468 Best: 0.849094 using {'batch_size': 32, 'lr': 0.02, 'max_epochs': 128, 
#                               'module__n_features': 8, 'module__n_neurons': 64, 'module__nonlin': Tanh()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_diab= MLPClassifier(batch_size= 32,
                            learning_rate='constant', learning_rate_init= 0.02,
                            max_iter=128,
                            hidden_layer_sizes=(64,),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_diab.fit(train_diab, labels_train_diab.values.ravel())

acc_nn1_diab= sklearn.metrics.accuracy_score(labels_test_diab.values.ravel(), nn1_model_diab.predict(test_diab))
acc_nn1_diab

0.7532467532467533

In [55]:
# 2024-05-31 17:04:14,821 Best: 0.849508 using {'batch_size': 32, 'lr': 0.001, 'max_epochs': 128, 
#                               'module__n_features': 8, 'module__n_neurons': 128, 'module__nonlin': ReLU()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_diab= MLPClassifier(batch_size= 32,
                            learning_rate='constant', learning_rate_init= 0.001,
                            max_iter=512,
                            hidden_layer_sizes=(128, 128),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_diab.fit(train_diab, labels_train_diab.values.ravel())

acc_nn2_diab= sklearn.metrics.accuracy_score(labels_test_diab.values.ravel(), nn2_model_diab.predict(test_diab))
acc_nn2_diab

0.7142857142857143

In [62]:
# 2024-06-03 16:05:00,462 Best: 0.852499 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 16, 
#                               'module__n_features': 8, 'module__n_neurons': 64, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_diab= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(64, 64, 64),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_diab.fit(train_diab, labels_train_diab.values.ravel())

acc_nn3_diab= sklearn.metrics.accuracy_score(labels_test_diab.values.ravel(), nn3_model_diab.predict(test_diab))
acc_nn3_diab

0.7597402597402597

In [64]:
# definitions ---- 126 samples from test dataset

# get n and m parameters from train and labels_train
n_diab, m_diab= texp.get_n_m_sizes(test_diab.loc[0:125], labels_test_diab[0:125])

# conversion of train_diab and labels_train_diab data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_diab.values)
tn_lb_tr= torch.from_numpy(labels_train_diab.values.ravel().astype(int))


# define a descriptor to synthetic data
descriptor_diab= dict()

h_min_dist_diab= texp.get_minimum_distance(train_diab)

# T-Exp explanation settings
descriptor_diab['h_min']= h_min_dist_diab
descriptor_diab['h_max']= 1
descriptor_diab['jacobian_eps']= 1e-3
descriptor_diab['max_itr']= 30
descriptor_diab['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_diab['num_samples']= 30
descriptor_diab['num_perts']= 10   # RIS/ROS
descriptor_diab['pert_max_distance']= (h_min_dist_diab/2)
descriptor_diab['num_runs']= 10    # RES
descriptor_diab['feature_metadata']= ['c'] * n_diab
descriptor_diab['p_norm']= 2
descriptor_diab['eps_norm']= 1e-6
descriptor_diab['top_k']= 0
descriptor_diab['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_diab['top_k'])

In [ ]:
# ---- 126 samples from test dataset

# evaluate relative input/output stability -- using the 2 first instances from train_diab
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_diab= stab.relative_stability(nn1_model_diab, test_diab[0:125], labels_test_diab[0:125], perturbation, 
                                        descriptor_diab, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

RIS/ROS --
Instance 0 out 125


In [ ]:
ris_ros_nn1_diab

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_diab
start= time.time()
print('RES --')
res_nn1_diab= stab.run_stability(nn1_model_diab, test_diab[0:125], labels_test_diab[0:125], 
                               descriptor_diab, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn1_diab

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_diab
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_diab= stab.relative_stability(nn2_model_diab, test_diab[0:125], labels_test_diab[0:125], perturbation, 
                                        descriptor_diab, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn2_diab

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_diab
start= time.time()
print('RES --')
res_nn2_diab= stab.run_stability(nn2_model_diab, test_diab[0:125], labels_test_diab[0:125], 
                               descriptor_diab, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn2_diab

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_diab
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_diab= stab.relative_stability(nn3_model_diab, test_diab[0:125], labels_test_diab[0:125], perturbation, 
                                        descriptor_diab, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn3_diab

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_diab
start= time.time()
print('RES --')
res_nn3_diab= stab.run_stability(nn3_model_diab, test_diab[0:125], labels_test_diab[0:125], 
                               descriptor_diab, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn3_diab

# 6. German Credit

In [34]:
ger_path= 'data/german/processed/'

train_ger= pd.read_csv(ger_path + 'X_train.csv')
test_ger = pd.read_csv(ger_path + 'X_test.csv')
labels_train_ger= pd.read_csv(ger_path + 'y_train.csv')
labels_test_ger = pd.read_csv(ger_path + 'y_test.csv')

In [40]:
# 2024-05-29 17:41:34,069 Best: 0.668612 using {'batch_size': 128, 'lr': 0.01, 'max_epochs': 128, 
#                               'module__n_features': 23, 'module__n_neurons': 16, 'module__nonlin': Tanh()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_ger= MLPClassifier(batch_size= 128,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=512,
                            hidden_layer_sizes=(16,),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_ger.fit(train_ger, labels_train_ger.values.ravel())

acc_nn1_ger= sklearn.metrics.accuracy_score(labels_test_ger.values.ravel(), nn1_model_ger.predict(test_ger))
acc_nn1_ger

0.63

In [45]:
# 2024-05-31 17:28:51,852 Best: 0.675973 using {'batch_size': 32, 'lr': 0.02, 'max_epochs': 16, 
#                               'module__n_features': 23, 'module__n_neurons': 256, 'module__nonlin': Tanh()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_ger= MLPClassifier(batch_size= 32,
                            learning_rate='constant', learning_rate_init= 0.02,
                            max_iter=32,
                            hidden_layer_sizes=(256, 256),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_ger.fit(train_ger, labels_train_ger.values.ravel())

acc_nn2_ger= sklearn.metrics.accuracy_score(labels_test_ger.values.ravel(), nn2_model_ger.predict(test_ger))
acc_nn2_ger

0.66

In [ ]:
# 2024-06-03 16:27:03,452 Best: 0.668002 using {'batch_size': 64, 'lr': 0.02, 'max_epochs': 32, 
#                               'module__n_features': 23, 'module__n_neurons': 16, 'module__nonlin': Tanh()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_ger= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.02,
                            max_iter=32,
                            hidden_layer_sizes=(16, 16, 16),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_ger.fit(train_ger, labels_train_ger.values.ravel())

acc_nn3_ger= sklearn.metrics.accuracy_score(labels_test_ger.values.ravel(), nn3_model_ger.predict(test_ger))
acc_nn3_ger

In [ ]:
# definitions ---- 154 samples from test dataset

# get n and m parameters from train and labels_train
n_ger, m_ger= texp.get_n_m_sizes(test_ger.loc[0:125], labels_test_ger[0:125])

# conversion of train_ger and labels_train_ger data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_ger.values)
tn_lb_tr= torch.from_numpy(labels_train_ger.values.ravel().astype(int))


# define a descriptor to synthetic data
descriptor_ger= dict()

h_min_dist_ger= texp.get_minimum_distance(train_ger)

# T-Exp explanation settings
descriptor_ger['h_min']= h_min_dist_ger
descriptor_ger['h_max']= 1
descriptor_ger['jacobian_eps']= 1e-3
descriptor_ger['max_itr']= 30
descriptor_ger['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_ger['num_samples']= 30
descriptor_ger['num_perts']= 10   # RIS/ROS
descriptor_ger['pert_max_distance']= (h_min_dist_ger/2)
descriptor_ger['num_runs']= 10    # RES
descriptor_ger['feature_metadata']= ['c'] * n_ger
descriptor_ger['p_norm']= 2
descriptor_ger['eps_norm']= 1e-6
descriptor_ger['top_k']= 0
descriptor_ger['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_ger['top_k'])

In [ ]:
# ---- 154 samples from test dataset

# evaluate relative input/output stability -- using the 2 first instances from train_ger
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_ger= stab.relative_stability(nn1_model_ger, test_ger[0:125], labels_test_ger[0:125], perturbation, 
                                        descriptor_ger, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn1_ger

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ger
start= time.time()
print('RES --')
res_nn1_ger= stab.run_stability(nn1_model_ger, test_ger[0:125], labels_test_ger[0:125], 
                               descriptor_ger, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn1_ger

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_ger
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_ger= stab.relative_stability(nn2_model_ger, test_ger[0:125], labels_test_ger[0:125], perturbation, 
                                        descriptor_ger, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn2_ger

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ger
start= time.time()
print('RES --')
res_nn2_ger= stab.run_stability(nn2_model_ger, test_ger[0:125], labels_test_ger[0:125], 
                               descriptor_ger, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn2_ger

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_ger
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_ger= stab.relative_stability(nn3_model_ger, test_ger[0:125], labels_test_ger[0:125], perturbation, 
                                        descriptor_ger, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn3_ger

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ger
start= time.time()
print('RES --')
res_nn3_ger= stab.run_stability(nn3_model_ger, test_ger[0:125], labels_test_ger[0:125], 
                               descriptor_ger, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn3_ger

# 7. HELOC

In [11]:
hel_path= 'data/heloc/processed/'

train_hel= pd.read_csv(hel_path + 'X_train.csv')
test_hel = pd.read_csv(hel_path + 'X_test.csv')
labels_train_hel= pd.read_csv(hel_path + 'y_train.csv')
labels_test_hel = pd.read_csv(hel_path + 'y_test.csv')

In [ ]:
# 2024-05-30 09:37:27,988 Best: 0.802750 using {'batch_size': 16, 'lr': 0.001, 'max_epochs': 128, 
#                               'module__n_features': 37, 'module__n_neurons': 256, 'module__nonlin': Tanh()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_hel= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.001,
                            max_iter=128,
                            hidden_layer_sizes=(256,),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_hel.fit(train_hel, labels_train_hel.values.ravel())

acc_nn1_hel= sklearn.metrics.accuracy_score(labels_test_hel.values.ravel(), nn1_model_hel.predict(test_hel))
acc_nn1_hel

In [ ]:
# 2024-06-01 04:05:45,253 Best: 0.802888 using {'batch_size': 16, 'lr': 0.001, 'max_epochs': 128, 
#                               'module__n_features': 37, 'module__n_neurons': 16, 'module__nonlin': Tanh()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_hel= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.001,
                            max_iter=128,
                            hidden_layer_sizes=(16, 16),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_hel.fit(train_hel, labels_train_hel.values.ravel())

acc_nn2_hel= sklearn.metrics.accuracy_score(labels_test_hel.values.ravel(), nn2_model_hel.predict(test_hel))
acc_nn2_hel

In [ ]:
# 2024-06-04 04:15:54,809 Best: 0.802922 using {'batch_size': 64, 'lr': 0.001, 'max_epochs': 16, 
#                               'module__n_features': 37, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_hel= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.001,
                            max_iter=16,
                            hidden_layer_sizes=(256, 256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_hel.fit(train_hel, labels_train_hel.values.ravel())

acc_nn3_hel= sklearn.metrics.accuracy_score(labels_test_hel.values.ravel(), nn3_model_hel.predict(test_hel))
acc_nn3_hel

In [ ]:
# definitions ---- 499 samples from test dataset

# get n and m parameters from train and labels_train
n_hel, m_hel= texp.get_n_m_sizes(test_hel.loc[0:498], labels_test_hel[0:498])

# conversion of train_hel and labels_train_hel data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_hel.values)
tn_lb_tr= torch.from_numpy(labels_train_hel.values.ravel().astype(int))


# define a descriptor to synthetic data
descriptor_hel= dict()

h_min_dist_hel= texp.get_minimum_distance(train_hel)

# T-Exp explanation settings
descriptor_hel['h_min']= h_min_dist_hel
descriptor_hel['h_max']= 1
descriptor_hel['jacobian_eps']= 1e-3
descriptor_hel['max_itr']= 30
descriptor_hel['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_hel['num_samples']= 30
descriptor_hel['num_perts']= 10   # RIS/ROS
descriptor_hel['pert_max_distance']= (h_min_dist_hel/2)
descriptor_hel['num_runs']= 10    # RES
descriptor_hel['feature_metadata']= ['c'] * n_hel
descriptor_hel['p_norm']= 2
descriptor_hel['eps_norm']= 1e-6
descriptor_hel['top_k']= 0
descriptor_hel['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_hel['top_k'])

In [ ]:
# ---- 499 samples from test dataset

# evaluate relative input/output stability -- using the 2 first instances from train_hel
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_hel= stab.relative_stability(nn1_model_hel, test_hel[0:498], labels_test_hel[0:498], perturbation, 
                                        descriptor_hel, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn1_hel

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_hel
start= time.time()
print('RES --')
res_nn1_hel= stab.run_stability(nn1_model_hel, test_hel[0:498], labels_test_hel[0:498], 
                               descriptor_hel, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn1_hel

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_hel
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_hel= stab.relative_stability(nn2_model_hel, test_hel[0:498], labels_test_hel[0:498], perturbation, 
                                        descriptor_hel, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn2_hel

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_hel
start= time.time()
print('RES --')
res_nn2_hel= stab.run_stability(nn2_model_hel, test_hel[0:498], labels_test_hel[0:498], 
                               descriptor_hel, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn2_hel

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_hel
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_hel= stab.relative_stability(nn3_model_hel, test_hel[0:498], labels_test_hel[0:498], perturbation, 
                                        descriptor_hel, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn3_hel

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_hel
start= time.time()
print('RES --')
res_nn3_hel= stab.run_stability(nn3_model_hel, test_hel[0:498], labels_test_hel[0:498], 
                               descriptor_hel, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn3_hel

# 8. HIGGS

In [12]:
hig_path= 'data/higgs/processed/'

train_hig= pd.read_csv(hig_path + 'X_train.csv')
test_hig = pd.read_csv(hig_path + 'X_test.csv')
labels_train_hig= pd.read_csv(hig_path + 'y_train.csv')
labels_test_hig = pd.read_csv(hig_path + 'y_test.csv')

In [ ]:
# 2024-05-29 23:24:10,338 Best: 0.908399 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_hig= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=256,
                            hidden_layer_sizes=(256,),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_hig.fit(train_hig, labels_train_hig.values.ravel())

acc_nn1_hig= sklearn.metrics.accuracy_score(labels_test_hig.values.ravel(), nn1_model_hig.predict(test_hig))
acc_nn1_hig

In [ ]:
# 2024-05-31 17:29:14,888 Best: 0.912651 using {'batch_size': 64, 'lr': 0.01, 'max_epochs': 64, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_hig= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_hig.fit(train_hig, labels_train_hig.values.ravel())

acc_nn2_hig= sklearn.metrics.accuracy_score(labels_test_hig.values.ravel(), nn2_model_hig.predict(test_hig))
acc_nn2_hig

In [ ]:
# 2024-06-03 16:27:56,409 Best: 0.905554 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_hig= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_hig.fit(train_hig, labels_train_hig.values.ravel())

acc_nn3_hig= sklearn.metrics.accuracy_score(labels_test_hig.values.ravel(), nn3_model_hig.predict(test_hig))
acc_nn3_hig

In [ ]:
# definitions ---- 644 samples from test dataset

# get n and m parameters from train and labels_train
n_hig, m_hig= texp.get_n_m_sizes(test_hig.loc[0:643], labels_test_hig[0:643])

# conversion of train_hig and labels_train_hig data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_hig.values)
tn_lb_tr= torch.from_numpy(labels_train_hig.values.ravel().astype(int))


# define a descriptor to synthetic data
descriptor_hig= dict()

h_min_dist_hig= texp.get_minimum_distance(train_hig)

# T-Exp explanation settings
descriptor_hig['h_min']= h_min_dist_hig
descriptor_hig['h_max']= 1
descriptor_hig['jacobian_eps']= 1e-3
descriptor_hig['max_itr']= 30
descriptor_hig['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_hig['num_samples']= 30
descriptor_hig['num_perts']= 10   # RIS/ROS
descriptor_hig['pert_max_distance']= (h_min_dist_hig/2)
descriptor_hig['num_runs']= 10    # RES
descriptor_hig['feature_metadata']= ['c'] * n_hig
descriptor_hig['p_norm']= 2
descriptor_hig['eps_norm']= 1e-6
descriptor_hig['top_k']= 0
descriptor_hig['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_hig['top_k'])

In [ ]:
# ---- 644 samples from test dataset

# evaluate relative input/output stability -- using the 2 first instances from train_hig
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_hig= stab.relative_stability(nn1_model_hig, test_hig[0:643], labels_test_hig[0:643], perturbation, 
                                        descriptor_hig, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn1_hig

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_hig
start= time.time()
print('RES --')
res_nn1_hig= stab.run_stability(nn1_model_hig, test_hig[0:643], labels_test_hig[0:643], 
                               descriptor_hig, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn1_hig

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_hig
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_hig= stab.relative_stability(nn2_model_hig, test_hig[0:643], labels_test_hig[0:643], perturbation, 
                                        descriptor_hig, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn2_hig

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_hig
start= time.time()
print('RES --')
res_nn2_hig= stab.run_stability(nn2_model_hig, test_hig[0:643], labels_test_hig[0:643], 
                               descriptor_hig, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn2_hig

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_hig
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_hig= stab.relative_stability(nn3_model_hig, test_hig[0:643], labels_test_hig[0:643], perturbation, 
                                        descriptor_hig, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn3_hig

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_hig
start= time.time()
print('RES --')
res_nn3_hig= stab.run_stability(nn3_model_hig, test_hig[0:643], labels_test_hig[0:643], 
                               descriptor_hig, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn3_hig

# 9. Independent

In [5]:
indep_path= 'data/independent/processed/'

train_indep= pd.read_csv(indep_path + 'X_train.csv')
test_indep = pd.read_csv(indep_path + 'X_test.csv')
labels_train_indep= pd.read_csv(indep_path + 'y_train.csv')
labels_test_indep = pd.read_csv(indep_path + 'y_test.csv')

In [6]:
# 2024-05-29 17:02:04,589 Best: 0.997288 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 32, 
#                               'module__n_features': 6, 'module__n_neurons': 256, 'module__nonlin': Tanh()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_indep= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=32,
                            hidden_layer_sizes=(256,),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_indep.fit(train_indep, labels_train_indep.values.ravel())

acc_nn1_indep= sklearn.metrics.accuracy_score(labels_test_indep.values.ravel(), nn1_model_indep.predict(test_indep))
acc_nn1_indep

Stochastic Optimizer: Maximum iterations (32) reached and the optimization hasn't converged yet.


0.9666666666666667

In [7]:
# 2024-05-31 16:17:24,671 Best: 0.997288 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 16, 
#                               'module__n_features': 6, 'module__n_neurons': 32, 'module__nonlin': Tanh()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_indep= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=16,
                            hidden_layer_sizes=(32, 32),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_indep.fit(train_indep, labels_train_indep.values.ravel())

acc_nn2_indep= sklearn.metrics.accuracy_score(labels_test_indep.values.ravel(), nn2_model_indep.predict(test_indep))
acc_nn2_indep

Stochastic Optimizer: Maximum iterations (16) reached and the optimization hasn't converged yet.


0.9666666666666667

In [8]:
# 2024-06-03 15:02:42,848 Best: 0.997667 using {'batch_size': 64, 'lr': 0.01, 'max_epochs': 16, 
#                               'module__n_features': 6, 'module__n_neurons': 32, 'module__nonlin': Tanh()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_indep= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=16,
                            hidden_layer_sizes=(32, 32, 32),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_indep.fit(train_indep, labels_train_indep.values.ravel())

acc_nn3_indep= sklearn.metrics.accuracy_score(labels_test_indep.values.ravel(), nn3_model_indep.predict(test_indep))
acc_nn3_indep

Stochastic Optimizer: Maximum iterations (16) reached and the optimization hasn't converged yet.


1.0

In [11]:
# definitions ---- 60 samples from test dataset

# get n and m parameters from train and labels_train
n_indep, m_indep= texp.get_n_m_sizes(test_indep, labels_test_indep)

# conversion of train_indep and labels_train_indep data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_indep.values)
tn_lb_tr= torch.from_numpy(labels_train_indep.values.ravel().astype(int))


# define a descriptor to synthetic data
descriptor_indep= dict()

h_min_dist_indep= texp.get_minimum_distance(train_indep)

# T-Exp explanation settings
descriptor_indep['h_min']= h_min_dist_indep
descriptor_indep['h_max']= 1
descriptor_indep['jacobian_eps']= 1e-3
descriptor_indep['max_itr']= 30
descriptor_indep['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_indep['num_samples']= 30
descriptor_indep['num_perts']= 10   # RIS/ROS
descriptor_indep['pert_max_distance']= (h_min_dist_indep/2)
descriptor_indep['num_runs']= 10    # RES
descriptor_indep['feature_metadata']= ['c'] * n_indep
descriptor_indep['p_norm']= 2
descriptor_indep['eps_norm']= 1e-6
descriptor_indep['top_k']= 0
descriptor_indep['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_indep['top_k'])

In [32]:
# ---- 60 samples from test dataset

# evaluate relative input/output stability -- using the 2 first instances from train_indep
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_indep= stab.relative_stability(nn1_model_indep, test_indep, labels_test_indep, perturbation, 
                                        descriptor_indep, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

RIS/ROS --
Instance 0 out 60


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


...

  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]


--- 703.41 seconds ---


Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


In [33]:
ris_ros_nn1_indep

`should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.


{'t_exp_ris_max': 206.34042617681843,
 'std(t_exp_ris_max)': 27.157808048256303,
 'shap_ris_max': 21792.502753060042,
 'std(shap_ris_max)': 2787.0830818686995,
 'lime_ris_max': 33.99804611037021,
 'std(lime_ris_max)': 7.231553593636686,
 't_exp_ris_mean': 3.85425748275405,
 'std(t_exp_ris_mean)': 11.330767925124785,
 'shap_ris_mean': 126.08329919304222,
 'std(shap_ris_mean)': 889.909839155565,
 'lime_ris_mean': 2.1350581627862213,
 'std(lime_ris_mean)': 3.9262423410240643,
 't_exp_ros_max': 8464.133293312612,
 'std(t_exp_ros_max)': 1525.9270148199305,
 'shap_ros_max': 212372.66629990033,
 'std(shap_ros_max)': 28286.878543494557,
 'lime_ros_max': 22253.940655431226,
 'std(lime_ros_max)': 2944.611927582192,
 't_exp_ros_mean': 101.65885776089065,
 'std(t_exp_ros_mean)': 370.85292080901564,
 'shap_ros_mean': 693.0197108640417,
 'std(shap_ros_mean)': 3871.3361858776957,
 'lime_ros_mean': 154.15000566268614,
 'std(lime_ros_mean)': 940.6050111933146,
 'shap_kernel_ris_max': 379.7427255918629,

In [14]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_indep
start= time.time()
print('RES --')
res_nn1_indep= stab.run_stability(nn1_model_indep, test_indep, labels_test_indep, 
                               descriptor_indep, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

RES --
Instance 0 out 60


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


...

  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]


--- 601.64 seconds ---


Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


In [15]:
res_nn1_indep

`should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.


{'texp_res': 4.070144838902081e-15,
 'shap_res': 1.161115229807228e-16,
 'shap_kernel_res': 1.1443916996305594e-16,
 'shap_exact_res': 1.161115229807228e-16,
 'lime_res': 7.343435057440258e-17,
 'itGd_res': 2.8087096313882582e-15,
 'iXGd_res': 1.4305164e-06,
 'dLif_res': 1.4305115e-06,
 'lwrp_res': 1.4305891e-06,
 'smoothG_res': 0.7703470999481264,
 'vanillaG_res': 2.3363957e-06,
 'GuidBprop_res': 2.3363957e-06,
 'occlusion_res': 1.2619445e-06}

In [16]:
# evaluate relative input/output stability -- using the 2 first instances from train_indep
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_indep= stab.relative_stability(nn2_model_indep, test_indep, labels_test_indep, perturbation, 
                                        descriptor_indep, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

RIS/ROS --
Instance 0 out 60


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


...

  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]


--- 692.15 seconds ---


Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


In [17]:
ris_ros_nn2_indep

`should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.


{'t_exp_ris_max': 373.00339353464375,
 'std(t_exp_ris_max)': 47.23694843396112,
 'shap_ris_max': 10910.741568374104,
 'std(shap_ris_max)': 1838.984604460384,
 'lime_ris_max': 834.2752302136252,
 'std(lime_ris_max)': 106.62739915259883,
 't_exp_ris_mean': 5.004472944111258,
 'std(t_exp_ris_mean)': 17.781705252851243,
 'shap_ris_mean': 167.677522163775,
 'std(shap_ris_mean)': 685.9062788593199,
 'lime_ris_mean': 8.241422027202237,
 'std(lime_ris_mean)': 48.00262352636836,
 't_exp_ros_max': 322.20252214159,
 'std(t_exp_ros_max)': 54.79309285860839,
 'shap_ros_max': 19632.427645927626,
 'std(shap_ros_max)': 3414.3663321317417,
 'lime_ros_max': 631.9089387725204,
 'std(lime_ros_max)': 132.72313102779188,
 't_exp_ros_mean': 4.919506082118588,
 'std(t_exp_ros_mean)': 10.246548648135322,
 'shap_ros_mean': 194.55191858522832,
 'std(shap_ros_mean)': 708.5628696147824,
 'lime_ros_mean': 8.261351630253602,
 'std(lime_ros_mean)': 27.139402448720332,
 'shap_kernel_ris_max': 140.9661418422737,
 'std(

In [18]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_indep
start= time.time()
print('RES --')
res_nn2_indep= stab.run_stability(nn2_model_indep, test_indep, labels_test_indep, 
                               descriptor_indep, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

RES --
Instance 0 out 60


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


...

  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]


--- 622.18 seconds ---


Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


In [19]:
res_nn2_indep

`should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.


{'texp_res': 5.402667531705947e-15,
 'shap_res': 1.2098374922832816e-16,
 'shap_kernel_res': 1.1775693440128312e-16,
 'shap_exact_res': 1.2098374922832816e-16,
 'lime_res': 6.800912535878896e-17,
 'itGd_res': 1.4043333874306805e-15,
 'iXGd_res': 1.1680315e-06,
 'dLif_res': 1.1198696e-06,
 'lwrp_res': 1.1246182e-06,
 'smoothG_res': 0.5532251739955422,
 'vanillaG_res': 2.9010996e-06,
 'GuidBprop_res': 2.9010996e-06,
 'occlusion_res': 8.678567e-07}

In [20]:
# evaluate relative input/output stability -- using the 2 first instances from train_indep
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_indep= stab.relative_stability(nn3_model_indep, test_indep, labels_test_indep, perturbation, 
                                        descriptor_indep, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

RIS/ROS --
Instance 0 out 60


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


...

  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]


--- 726.87 seconds ---


Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


In [21]:
ris_ros_nn3_indep

`should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.


{'t_exp_ris_max': 31.71689397067701,
 'std(t_exp_ris_max)': 4.5200027420063025,
 'shap_ris_max': 40099.64828787683,
 'std(shap_ris_max)': 5414.285659233127,
 'lime_ris_max': 0.8891024724543918,
 'std(lime_ris_max)': 0.17081009409969775,
 't_exp_ris_mean': 2.692597931619889,
 'std(t_exp_ris_mean)': 1.8369968522281748,
 'shap_ris_mean': 425.59395273028866,
 'std(shap_ris_mean)': 2581.1345962736727,
 'lime_ris_mean': 0.2857316723561576,
 'std(lime_ris_mean)': 0.08935926731495729,
 't_exp_ros_max': 1372.8567153272957,
 'std(t_exp_ros_max)': 179.9606422969648,
 'shap_ros_max': 57379.40041398179,
 'std(shap_ros_max)': 8893.386521711727,
 'lime_ros_max': 81.7109657926291,
 'std(lime_ros_max)': 12.49211248772398,
 't_exp_ros_mean': 7.590915535815384,
 'std(t_exp_ros_mean)': 25.23898678786119,
 'shap_ros_mean': 429.24841493301426,
 'std(shap_ros_mean)': 2257.7357997938707,
 'lime_ros_mean': 0.7955566093992672,
 'std(lime_ros_mean)': 2.1980410859374304,
 'shap_kernel_ris_max': 675.9950735631483,

In [22]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_indep
start= time.time()
print('RES --')
res_nn3_indep= stab.run_stability(nn3_model_indep, test_indep, labels_test_indep, 
                               descriptor_indep, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

RES --
Instance 0 out 60


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


...

  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]


--- 637.54 seconds ---


Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


In [23]:
res_nn3_indep

`should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.


{'texp_res': 4.39625888431457e-15,
 'shap_res': 1.1862261865335482e-16,
 'shap_kernel_res': 1.2412670766236366e-16,
 'shap_exact_res': 1.1862261865335482e-16,
 'lime_res': 8.778797778219434e-17,
 'itGd_res': 1.5423867269772919e-15,
 'iXGd_res': 9.684609e-07,
 'dLif_res': 9.684609e-07,
 'lwrp_res': 1.0115243e-06,
 'smoothG_res': 0.45614080121455153,
 'vanillaG_res': 1.922192e-06,
 'GuidBprop_res': 1.922192e-06,
 'occlusion_res': 9.629425e-07}

# 10. LSA - Law School Admission

In [14]:
lsa_path= 'data/law_school_admission/processed/'

train_lsa= pd.read_csv(lsa_path + 'X_train.csv')
test_lsa = pd.read_csv(lsa_path + 'X_test.csv')
labels_train_lsa= pd.read_csv(lsa_path + 'y_train.csv')
labels_test_lsa = pd.read_csv(lsa_path + 'y_test.csv')

In [ ]:
# 2024-05-29 23:24:10,338 Best: 0.908399 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_lsa= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=256,
                            hidden_layer_sizes=(256,),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_lsa.fit(train_lsa, labels_train_lsa.values.ravel())

acc_nn1_lsa= sklearn.metrics.accuracy_score(labels_test_lsa.values.ravel(), nn1_model_lsa.predict(test_lsa))
acc_nn1_lsa

In [ ]:
# 2024-05-31 17:29:14,888 Best: 0.912651 using {'batch_size': 64, 'lr': 0.01, 'max_epochs': 64, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_lsa= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_lsa.fit(train_lsa, labels_train_lsa.values.ravel())

acc_nn2_lsa= sklearn.metrics.accuracy_score(labels_test_lsa.values.ravel(), nn2_model_lsa.predict(test_lsa))
acc_nn2_lsa

In [ ]:
# 2024-06-03 16:27:56,409 Best: 0.905554 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_lsa= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_lsa.fit(train_lsa, labels_train_lsa.values.ravel())

acc_nn3_lsa= sklearn.metrics.accuracy_score(labels_test_lsa.values.ravel(), nn3_model_lsa.predict(test_lsa))
acc_nn3_lsa

In [ ]:
# definitions ---- 574 samples from test dataset

# get n and m parameters from train and labels_train
n_lsa, m_lsa= texp.get_n_m_sizes(test_lsa.loc[0:573], labels_test_lsa[0:573])

# conversion of train_lsa and labels_train_lsa data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_lsa.values)
tn_lb_tr= torch.from_numpy(labels_train_lsa.values.ravel().astype(int))


# define a descriptor to synthetic data
descriptor_lsa= dict()

h_min_dist_lsa= texp.get_minimum_distance(train_lsa)

# T-Exp explanation settings
descriptor_lsa['h_min']= h_min_dist_lsa
descriptor_lsa['h_max']= 1
descriptor_lsa['jacobian_eps']= 1e-3
descriptor_lsa['max_itr']= 30
descriptor_lsa['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_lsa['num_samples']= 30
descriptor_lsa['num_perts']= 10   # RIS/ROS
descriptor_lsa['pert_max_distance']= (h_min_dist_lsa/2)
descriptor_lsa['num_runs']= 10    # RES
descriptor_lsa['feature_metadata']= ['c'] * n_lsa
descriptor_lsa['p_norm']= 2
descriptor_lsa['eps_norm']= 1e-6
descriptor_lsa['top_k']= 0
descriptor_lsa['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_lsa['top_k'])

In [ ]:
# ---- 574 samples from test dataset

# evaluate relative input/output stability -- using the 2 first instances from train_lsa
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_lsa= stab.relative_stability(nn1_model_lsa, test_lsa[0:573], labels_test_lsa[0:573], perturbation, 
                                        descriptor_lsa, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn1_lsa

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_lsa
start= time.time()
print('RES --')
res_nn1_lsa= stab.run_stability(nn1_model_lsa, test_lsa[0:573], labels_test_lsa[0:573], 
                               descriptor_lsa, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn1_lsa

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_lsa
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_lsa= stab.relative_stability(nn2_model_lsa, test_lsa[0:573], labels_test_lsa[0:573], perturbation, 
                                        descriptor_lsa, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn2_lsa

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_lsa
start= time.time()
print('RES --')
res_nn2_lsa= stab.run_stability(nn2_model_lsa, test_lsa[0:573], labels_test_lsa[0:573], 
                               descriptor_lsa, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn2_lsa

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_lsa
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_lsa= stab.relative_stability(nn3_model_lsa, test_lsa[0:573], labels_test_lsa[0:573], perturbation, 
                                        descriptor_lsa, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
ris_ros_nn3_lsa

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_lsa
start= time.time()
print('RES --')
res_nn3_lsa= stab.run_stability(nn3_model_lsa, test_lsa[0:573], labels_test_lsa[0:573], 
                               descriptor_lsa, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

In [ ]:
res_nn3_lsa